In [ ]:
# 1. Instalación de dependencias.
!pip install -q ultralytics transformers accelerate sentencepiece sacremoses openpyxl

In [ ]:
# 2. Importación de librerías.
import os
import re
import cv2
import torch
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from ultralytics import YOLO
from transformers import (
    Blip2Processor,
    Blip2ForConditionalGeneration,
    MarianMTModel,
    MarianTokenizer
)
from google.colab import files
from IPython.display import display

print("Librerías importadas correctamente.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Librerías importadas correctamente.


In [ ]:
# 3. Verificación del entorno y configuración del dispositivo.
import platform

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Dispositivo:", dispositivo)
if dispositivo == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No se detectó GPU. BLIP-2 puede requerir mucha memoria en CPU.")

Python: 3.13.16
PyTorch: 2.14.1+cu130
Dispositivo: cuda
GPU: Tesla T4


In [ ]:
# 4. Carga de los pesos YOLOv8n entrenados en TG1.
print("Selecciona el archivo best.pt")
subidos_pesos = files.upload()
if not subidos_pesos:
    raise ValueError("No se cargó el archivo de pesos.")

nombre_pesos = next(iter(subidos_pesos))
ruta_pesos = os.path.join("/content", nombre_pesos)
modelo_yolo = YOLO(ruta_pesos)
print("YOLOv8n cargado.")
print("Clases del modelo:", modelo_yolo.names)

Selecciona el archivo best.pt


Saving best.pt to best.pt
YOLOv8n cargado.
Clases del modelo: {0: 'knife', 1: 'pistol'}


In [ ]:
# 5. Carga del modelo BLIP-2.
nombre_modelo_blip = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(nombre_modelo_blip)

tipo_dato = torch.float16 if dispositivo == "cuda" else torch.float32
modelo_blip2 = Blip2ForConditionalGeneration.from_pretrained(
    nombre_modelo_blip,
    torch_dtype=tipo_dato,
    device_map="auto" if dispositivo == "cuda" else None
)
if dispositivo == "cpu":
    modelo_blip2 = modelo_blip2.to(dispositivo)

print("BLIP-2 cargado correctamente.")

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

BLIP-2 cargado correctamente.


In [ ]:
# 6. Carga del modelo de traducción inglés-español.
nombre_modelo_traduccion = "Helsinki-NLP/opus-mt-en-es"
tokenizer_traduccion = MarianTokenizer.from_pretrained(nombre_modelo_traduccion)
traductor = MarianMTModel.from_pretrained(nombre_modelo_traduccion).to(dispositivo)
print("Modelo de traducción cargado correctamente.")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  312MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Modelo de traducción cargado correctamente.


In [ ]:
# 7. Carga del video.
print("Selecciona el video que deseas analizar")
subidos_video = files.upload()
if not subidos_video:
    raise ValueError("No se cargó ningún video.")

video_nombre = next(iter(subidos_video))
video_path = os.path.join("/content", video_nombre)
print("Video:", video_nombre)
print("Ruta:", video_path)

Selecciona el video que deseas analizar


Saving video_sin_arma.webm to video_sin_arma.webm
Video: video_sin_arma.webm
Ruta: /content/video_sin_arma.webm


In [ ]:
# 8. Verificación de las características del video.
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError("No fue posible abrir el video.")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if fps <= 0 or total_frames <= 0:
    cap.release()
    raise ValueError("El video no tiene propiedades válidas.")

duracion = total_frames / fps
cap.release()
print("Duración:", round(duracion, 2), "segundos")
print("FPS:", round(fps, 3))
print("Fotogramas:", total_frames)
print("Resolución:", ancho, "x", alto)

Duración: 110.98 segundos
FPS: 13.093
Fotogramas: 1453
Resolución: 1270 x 720


In [ ]:
# 9. Parámetros generales del análisis.
umbral_histograma = 0.25
umbral_diferencia = 0.12
intervalo_respaldo = 8
separacion_minima = 3
intervalo_yolo = 2

print("Parámetros configurados.")

Parámetros configurados.


In [ ]:
# 10. Detección de objetos con YOLOv8n a intervalos regulares.
registros_yolo = []
cap = cv2.VideoCapture(video_path)

for segundo in range(0, int(duracion) + 1, intervalo_yolo):
    cap.set(cv2.CAP_PROP_POS_MSEC, segundo * 1000)
    ret, frame = cap.read()
    if not ret:
        continue

    resultado = modelo_yolo.predict(
        source=frame,
        device=0 if dispositivo == "cuda" else "cpu",
        verbose=False
    )[0]

    for caja in resultado.boxes:
        clase_id = int(caja.cls[0])
        registros_yolo.append({
            "Segundo": segundo,
            "Clase": modelo_yolo.names[clase_id],
            "Confianza": round(float(caja.conf[0]), 4)
        })

cap.release()
df_detecciones = pd.DataFrame(
    registros_yolo,
    columns=["Segundo", "Clase", "Confianza"]
)
print("Detección YOLO finalizada.")
print("Total de predicciones:", len(df_detecciones))
if df_detecciones.empty:
    print("YOLO no registró predicciones de knife o pistol en el muestreo.")
else:
    display(df_detecciones.head(10))

Detección YOLO finalizada.
Total de predicciones: 14


,Segundo,Clase,Confianza
0,14,pistol,0.3299
1,16,pistol,0.3310
2,18,pistol,0.3348
3,20,pistol,0.3287
4,20,pistol,0.3224
5,34,pistol,0.3974
6,54,pistol,0.3130
7,64,pistol,0.4990
8,68,pistol,0.2946
9,68,pistol,0.2569


In [ ]:
# 11. Selección automática de fotogramas.
# Se consideran: segundos con detecciones YOLO, cambios visuales y respaldo temporal.

output_dir = "/content/fotogramas_descripcion"
os.makedirs(output_dir, exist_ok=True)
for nombre in os.listdir(output_dir):
    ruta = os.path.join(output_dir, nombre)
    if os.path.isfile(ruta):
        os.remove(ruta)

segundos_yolo = sorted(df_detecciones["Segundo"].unique().tolist()) if not df_detecciones.empty else []
segundos_cambio = []
cap = cv2.VideoCapture(video_path)
frame_anterior = None
hist_anterior = None

for segundo in range(0, int(duracion) + 1):
    cap.set(cv2.CAP_PROP_POS_MSEC, segundo * 1000)
    ret, frame = cap.read()
    if not ret:
        continue

    reducido = cv2.resize(frame, (320, 180))
    hsv = cv2.cvtColor(reducido, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)

    if frame_anterior is not None:
        distancia_hist = cv2.compareHist(hist_anterior, hist, cv2.HISTCMP_BHATTACHARYYA)
        diferencia = np.mean(cv2.absdiff(frame_anterior, reducido)) / 255.0
        if distancia_hist >= umbral_histograma or diferencia >= umbral_diferencia:
            segundos_cambio.append(segundo)

    frame_anterior = reducido
    hist_anterior = hist

cap.release()
segundos_respaldo = list(range(0, int(duracion) + 1, intervalo_respaldo))

# Prioridad: detección YOLO > cambio visual > respaldo temporal.
candidatos = []
for segundo in segundos_yolo:
    candidatos.append((int(segundo), 3))
for segundo in segundos_cambio:
    candidatos.append((int(segundo), 2))
for segundo in segundos_respaldo:
    candidatos.append((int(segundo), 1))

candidatos.sort(key=lambda x: (-x[1], x[0]))
seleccionados = []
for segundo, prioridad in candidatos:
    if all(abs(segundo - existente) >= separacion_minima for existente in seleccionados):
        seleccionados.append(segundo)

seleccionados = sorted(set(seleccionados))

# Extraer y guardar los fotogramas originales seleccionados.
registros_fotogramas = []
cap = cv2.VideoCapture(video_path)
for segundo in seleccionados:
    cap.set(cv2.CAP_PROP_POS_MSEC, segundo * 1000)
    ret, frame = cap.read()
    if not ret:
        continue
    archivo = f"fotograma_{segundo}s.jpg"
    cv2.imwrite(os.path.join(output_dir, archivo), frame)
    registros_fotogramas.append({"Segundo": segundo, "Archivo": archivo})
cap.release()

df_fotogramas = pd.DataFrame(registros_fotogramas, columns=["Segundo", "Archivo"])
print("Fotogramas seleccionados:", len(df_fotogramas))
print("Candidatos por YOLO:", len(segundos_yolo))
print("Cambios visuales detectados:", len(segundos_cambio))
print("Muestreos temporales:", len(segundos_respaldo))
display(df_fotogramas.head(10))

Fotogramas seleccionados: 17
Candidatos por YOLO: 11
Cambios visuales detectados: 0
Muestreos temporales: 14


,Segundo,Archivo
0,0,fotograma_0s.jpg
1,8,fotograma_8s.jpg
2,14,fotograma_14s.jpg
3,18,fotograma_18s.jpg
4,24,fotograma_24s.jpg
5,34,fotograma_34s.jpg
6,40,fotograma_40s.jpg
7,48,fotograma_48s.jpg
8,54,fotograma_54s.jpg
9,64,fotograma_64s.jpg


In [ ]:
# 12. Filtrado de fotogramas con detecciones y generación de cajas delimitadoras.
# Se vuelve a ejecutar YOLO únicamente sobre los fotogramas seleccionados para guardar
# las anotaciones y asociar predicciones a cada fotograma.

output_boxes = "/content/fotogramas_con_detecciones"
os.makedirs(output_boxes, exist_ok=True)
for nombre in os.listdir(output_boxes):
    ruta = os.path.join(output_boxes, nombre)
    if os.path.isfile(ruta):
        os.remove(ruta)

registros_detecciones = []
for _, fila in df_fotogramas.iterrows():
    segundo = int(fila["Segundo"])
    archivo = fila["Archivo"]
    ruta_imagen = os.path.join(output_dir, archivo)
    imagen = cv2.imread(ruta_imagen)
    if imagen is None:
        print(f"No se pudo leer {archivo}; se omite.")
        continue

    resultado = modelo_yolo.predict(
        source=imagen,
        device=0 if dispositivo == "cuda" else "cpu",
        verbose=False
    )[0]

    if len(resultado.boxes) > 0:
        clases = [modelo_yolo.names[int(caja.cls[0])] for caja in resultado.boxes]
        confianzas = [round(float(caja.conf[0]), 4) for caja in resultado.boxes]
        archivo_anotado = f"fotograma_{segundo}s_anotado.jpg"
        cv2.imwrite(
            os.path.join(output_boxes, archivo_anotado),
            resultado.plot()
        )
        registros_detecciones.append({
            "Segundo": segundo,
            "Archivo": archivo,
            "Archivo_anotado": archivo_anotado,
            "Detecciones": ", ".join(clases),
            "Confianzas": ", ".join(map(str, confianzas)),
            "Cantidad": len(clases)
        })

df_fotogramas_con_detecciones = pd.DataFrame(
    registros_detecciones,
    columns=["Segundo", "Archivo", "Archivo_anotado", "Detecciones", "Confianzas", "Cantidad"]
)
print("Filtrado finalizado.")
print("Fotogramas seleccionados:", len(df_fotogramas))
print("Fotogramas con detecciones:", len(df_fotogramas_con_detecciones))
if df_fotogramas_con_detecciones.empty:
    print("No se detectaron las clases knife o pistol en los fotogramas seleccionados.")
else:
    display(df_fotogramas_con_detecciones)

Filtrado finalizado.
Fotogramas seleccionados: 17
Fotogramas con detecciones: 8


,Segundo,Archivo,Archivo_anotado,Detecciones,Confianzas,Cantidad
0,14,fotograma_14s.jpg,fotograma_14s_anotado.jpg,pistol,0.2792,1
1,18,fotograma_18s.jpg,fotograma_18s_anotado.jpg,pistol,0.3213,1
2,34,fotograma_34s.jpg,fotograma_34s_anotado.jpg,pistol,0.3741,1
3,54,fotograma_54s.jpg,fotograma_54s_anotado.jpg,pistol,0.3942,1
4,64,fotograma_64s.jpg,fotograma_64s_anotado.jpg,pistol,0.5267,1
5,68,fotograma_68s.jpg,fotograma_68s_anotado.jpg,"pistol, pistol","0.3219, 0.2819",2
6,72,fotograma_72s.jpg,fotograma_72s_anotado.jpg,pistol,0.407,1
7,92,fotograma_92s.jpg,fotograma_92s_anotado.jpg,pistol,0.3662,1


In [ ]:
# 13. Generación de descripciones BLIP-2 para TODOS los fotogramas seleccionados.
# Se usan las imágenes originales, sin cajas delimitadoras.

resultados_blip = []
for i, fila in df_fotogramas.iterrows():
    ruta = os.path.join(output_dir, fila["Archivo"])
    imagen = cv2.imread(ruta)
    if imagen is None:
        print(f"No se pudo leer {fila['Archivo']}; se omite.")
        continue
    imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
    entradas = processor(images=imagen, return_tensors="pt")
    dispositivo_blip = next(modelo_blip2.parameters()).device
    entradas = {k: v.to(dispositivo_blip) for k, v in entradas.items()}
    if "pixel_values" in entradas:
        entradas["pixel_values"] = entradas["pixel_values"].to(dtype=tipo_dato)

    with torch.no_grad():
        salida = modelo_blip2.generate(**entradas, max_new_tokens=100)
    descripcion = processor.decode(salida[0], skip_special_tokens=True)
    resultados_blip.append({
        "Segundo": int(fila["Segundo"]),
        "Archivo": fila["Archivo"],
        "Descripcion": descripcion
    })
    if (i + 1) % 5 == 0 or i + 1 == len(df_fotogramas):
        print(f"Procesados {i + 1} de {len(df_fotogramas)} fotogramas")

df_blip = pd.DataFrame(resultados_blip, columns=["Segundo", "Archivo", "Descripcion"])
print("Descripciones generadas:", len(df_blip))
display(df_blip)

Procesados 5 de 17 fotogramas
Procesados 10 de 17 fotogramas
Procesados 15 de 17 fotogramas
Procesados 17 de 17 fotogramas
Descripciones generadas: 17


,Segundo,Archivo,Descripcion
0,0,fotograma_0s.jpg,a view of a store with people shopping\n
1,8,fotograma_8s.jpg,a view from above of a store with people shopp...
2,14,fotograma_14s.jpg,a large store with people shopping in it\n
3,18,fotograma_18s.jpg,a large group of people in a store\n
4,24,fotograma_24s.jpg,a large group of people in a store\n
5,34,fotograma_34s.jpg,a view of a store with people shopping\n
6,40,fotograma_40s.jpg,a view from above of a store with people shopp...
7,48,fotograma_48s.jpg,a view of a store with people shopping\n
8,54,fotograma_54s.jpg,a video surveillance camera shows people shopp...
9,64,fotograma_64s.jpg,a surveillance camera shows people shopping in...


In [ ]:
# 14. Traducción de las descripciones al español.
def traducir_texto(texto):
    entradas = tokenizer_traduccion(
        str(texto), return_tensors="pt", truncation=True, max_length=512
    ).to(dispositivo)
    with torch.no_grad():
        salida = traductor.generate(**entradas, max_new_tokens=120)
    return tokenizer_traduccion.decode(salida[0], skip_special_tokens=True)

df_blip["Descripcion_es"] = df_blip["Descripcion"].apply(traducir_texto)
print("Traducción finalizada.")
display(df_blip[["Segundo", "Descripcion", "Descripcion_es"]])

[transformers] Both `max_new_tokens` (=120) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Traducción finalizada.


,Segundo,Descripcion,Descripcion_es
0,0,a view of a store with people shopping\n,una vista de una tienda con gente de compras
1,8,a view from above of a store with people shopp...,una vista desde arriba de una tienda con gente...
2,14,a large store with people shopping in it\n,una gran tienda con gente comprando en ella
3,18,a large group of people in a store\n,un gran grupo de personas en una tienda
4,24,a large group of people in a store\n,un gran grupo de personas en una tienda
5,34,a view of a store with people shopping\n,una vista de una tienda con gente de compras
6,40,a view from above of a store with people shopp...,una vista desde arriba de una tienda con gente...
7,48,a view of a store with people shopping\n,una vista de una tienda con gente de compras
8,54,a video surveillance camera shows people shopp...,una cámara de video vigilancia muestra a la ge...
9,64,a surveillance camera shows people shopping in...,una cámara de vigilancia muestra a la gente co...


In [ ]:
# 15. Integración de BLIP-2 y YOLOv8n.
# Se usa left join para conservar TODOS los fotogramas descritos por BLIP-2.
columnas_yolo = ["Segundo", "Detecciones", "Confianzas", "Cantidad"]
df_integrado = pd.merge(
    df_blip,
    df_fotogramas_con_detecciones[columnas_yolo],
    on="Segundo",
    how="left"
)

df_integrado["Detecciones"] = df_integrado["Detecciones"].fillna("Sin detecciones")
df_integrado["Confianzas"] = df_integrado["Confianzas"].fillna("No disponible")
df_integrado["Cantidad"] = df_integrado["Cantidad"].fillna(0).astype(int)
df_integrado = df_integrado.sort_values("Segundo").reset_index(drop=True)

print("Integración finalizada.")
print("Fotogramas integrados:", len(df_integrado))
print("Fotogramas sin detecciones YOLO:", (df_integrado["Cantidad"] == 0).sum())
display(df_integrado)

Integración finalizada.
Fotogramas integrados: 17
Fotogramas sin detecciones YOLO: 9


,Segundo,Archivo,Descripcion,Descripcion_es,Detecciones,Confianzas,Cantidad
0,0,fotograma_0s.jpg,a view of a store with people shopping\n,una vista de una tienda con gente de compras,Sin detecciones,No disponible,0
1,8,fotograma_8s.jpg,a view from above of a store with people shopp...,una vista desde arriba de una tienda con gente...,Sin detecciones,No disponible,0
2,14,fotograma_14s.jpg,a large store with people shopping in it\n,una gran tienda con gente comprando en ella,pistol,0.2792,1
3,18,fotograma_18s.jpg,a large group of people in a store\n,un gran grupo de personas en una tienda,pistol,0.3213,1
4,24,fotograma_24s.jpg,a large group of people in a store\n,un gran grupo de personas en una tienda,Sin detecciones,No disponible,0
5,34,fotograma_34s.jpg,a view of a store with people shopping\n,una vista de una tienda con gente de compras,pistol,0.3741,1
6,40,fotograma_40s.jpg,a view from above of a store with people shopp...,una vista desde arriba de una tienda con gente...,Sin detecciones,No disponible,0
7,48,fotograma_48s.jpg,a view of a store with people shopping\n,una vista de una tienda con gente de compras,Sin detecciones,No disponible,0
8,54,fotograma_54s.jpg,a video surveillance camera shows people shopp...,una cámara de video vigilancia muestra a la ge...,pistol,0.3942,1
9,64,fotograma_64s.jpg,a surveillance camera shows people shopping in...,una cámara de vigilancia muestra a la gente co...,pistol,0.5267,1


In [ ]:
# 16. Generación de contextos individuales.

def generar_contexto(fila):
    if fila["Cantidad"] == 0:
        texto_yolo = (
            "YOLOv8n no registró predicciones de las clases knife o pistol "
            "en este fotograma. Esto no demuestra que no existan armas en la escena."
        )
    else:
        texto_yolo = (
            f"YOLOv8n registró predicciones de las clases {fila['Detecciones']}, "
            f"con confianza(s) {fila['Confianzas']}. Son predicciones del detector "
            "y requieren verificación visual."
        )
    return (
        f"En el segundo {fila['Segundo']}, BLIP-2 describió visualmente la escena como: "
        f"{fila['Descripcion_es']}. {texto_yolo}"
    )

df_integrado["Contexto generado"] = df_integrado.apply(generar_contexto, axis=1)
print("Contextos individuales generados:", len(df_integrado))
display(df_integrado[["Segundo", "Descripcion_es", "Detecciones", "Contexto generado"]])

Contextos individuales generados: 17


,Segundo,Descripcion_es,Detecciones,Contexto generado
0,0,una vista de una tienda con gente de compras,Sin detecciones,"En el segundo 0, BLIP-2 describió visualmente ..."
1,8,una vista desde arriba de una tienda con gente...,Sin detecciones,"En el segundo 8, BLIP-2 describió visualmente ..."
2,14,una gran tienda con gente comprando en ella,pistol,"En el segundo 14, BLIP-2 describió visualmente..."
3,18,un gran grupo de personas en una tienda,pistol,"En el segundo 18, BLIP-2 describió visualmente..."
4,24,un gran grupo de personas en una tienda,Sin detecciones,"En el segundo 24, BLIP-2 describió visualmente..."
5,34,una vista de una tienda con gente de compras,pistol,"En el segundo 34, BLIP-2 describió visualmente..."
6,40,una vista desde arriba de una tienda con gente...,Sin detecciones,"En el segundo 40, BLIP-2 describió visualmente..."
7,48,una vista de una tienda con gente de compras,Sin detecciones,"En el segundo 48, BLIP-2 describió visualmente..."
8,54,una cámara de video vigilancia muestra a la ge...,pistol,"En el segundo 54, BLIP-2 describió visualmente..."
9,64,una cámara de vigilancia muestra a la gente co...,pistol,"En el segundo 64, BLIP-2 describió visualmente..."


In [ ]:
# 17. Generación del contexto general del video.


# 1. Preparar los datos.
df_contexto = df_integrado.copy()

df_contexto["Descripcion_es"] = (
    df_contexto["Descripcion_es"].fillna("").astype(str)
)

# Asociar las detecciones YOLO con cada fotograma
detecciones_por_segundo = (
    df_fotogramas_con_detecciones
    .groupby("Segundo")["Detecciones"]
    .apply(list)
    .to_dict()
)

df_contexto["Detecciones_YOLO"] = (
    df_contexto["Segundo"].map(detecciones_por_segundo)
)

df_contexto["Detecciones_YOLO"] = (
    df_contexto["Detecciones_YOLO"].apply(
        lambda x: x if isinstance(x, list) else []
    )
)

# 2. Contar las predicciones de YOLO.
conteo_clases = Counter()

for detecciones in df_fotogramas_con_detecciones["Detecciones"]:
    texto = str(detecciones).lower()

    for clase in ["knife", "pistol"]:
        conteo_clases[clase] += len(
            re.findall(rf"\b{clase}\b", texto)
        )

# 3. Identificar elementos recurrentes en BLIP-2.
palabras_vacias = {
    "this", "that", "there", "with", "from", "have", "being",
    "while", "where", "which", "their", "they", "them",
    "some", "what", "when", "into", "over", "under",
    "the", "and", "for", "are", "was", "were", "has",
    "una", "uno", "unos", "unas", "que", "con", "del",
    "las", "los", "por", "para", "como", "sobre", "entre",
    "desde", "donde", "esta", "este", "estos", "estas",
    "hay", "ser", "estar", "siendo", "tiene", "tienen",
    "puede", "pueden", "parece", "parecen", "muestra",
    "imagen", "foto", "fotos", "fotografia", "persona",
    "personas", "hombre", "mujer", "alguien", "algo",
    "está", "están", "estaba", "estaban", "palabras",
    "pie", "pies", "fuera", "dentro", "lado", "parte",
    "primer", "plano", "frente", "si", "tambien"
}

palabras = []

for descripcion in df_contexto["Descripcion_es"]:
    tokens = re.findall(
        r"\b[a-záéíóúñü]{4,}\b",
        descripcion.lower()
    )

    palabras.extend([
        palabra for palabra in tokens
        if palabra not in palabras_vacias
    ])

frecuencias = Counter(palabras)

elementos_recurrentes = [
    palabra
    for palabra, frecuencia in frecuencias.most_common(10)
    if frecuencia >= 2
]

# 4. Analizar las descripciones por segmentos temporales.
df_contexto = df_contexto.sort_values("Segundo")

tercio = duracion / 3

segmentos = {
    "inicial": df_contexto[df_contexto["Segundo"] < tercio],
    "intermedio": df_contexto[
        (df_contexto["Segundo"] >= tercio) &
        (df_contexto["Segundo"] < 2 * tercio)
    ],
    "final": df_contexto[df_contexto["Segundo"] >= 2 * tercio]
}

elementos_temporales = {}

for nombre, datos in segmentos.items():
    palabras_segmento = []

    for descripcion in datos["Descripcion_es"]:
        tokens = re.findall(
            r"\b[a-záéíóúñü]{4,}\b",
            descripcion.lower()
        )

        palabras_segmento.extend([
            palabra for palabra in tokens
            if palabra not in palabras_vacias
        ])

    conteo = Counter(palabras_segmento)

    elementos_temporales[nombre] = [
        palabra
        for palabra, frecuencia in conteo.most_common(5)
        if frecuencia >= 2
    ]

# 5. Relacionar las detecciones YOLO con las descripciones BLIP-2.
terminos_armas = {
    "knife": [
        "knife", "cuchillo", "cuchillos", "navaja", "navajas"
    ],
    "pistol": [
        "pistol", "pistola", "pistolas", "gun",
        "handgun", "arma de fuego", "revolver"
    ]
}

coincidencias = []
menciones_generales_armas = []

for _, fila in df_contexto.iterrows():

    descripcion = fila["Descripcion_es"].lower()
    detecciones = fila["Detecciones_YOLO"]

    clases_detectadas = []

    for elemento in detecciones:
        texto = str(elemento).lower()

        for clase in ["knife", "pistol"]:
            if re.search(rf"\b{clase}\b", texto):
                clases_detectadas.append(clase)

    # Buscar menciones generales de armas
    if re.search(r"\b(arma|armas|weapon|weapons)\b", descripcion):
        if clases_detectadas:
            menciones_generales_armas.append({
                "Segundo": fila["Segundo"],
                "Clases_YOLO": ", ".join(set(clases_detectadas)),
                "Mencion_BLIP2": "arma"
            })

    # Buscar menciones específicas según la clase detectada
    for clase in set(clases_detectadas):

        terminos = terminos_armas[clase]

        menciones = [
            termino for termino in terminos
            if re.search(
                rf"\b{re.escape(termino)}\b",
                descripcion
            )
        ]

        if menciones:
            coincidencias.append({
                "Segundo": fila["Segundo"],
                "Clase_YOLO": clase,
                "Mencion_BLIP2": ", ".join(menciones)
            })

# 6. Calcular cantidades de fotogramas.
total_fotogramas = len(df_fotogramas)

fotogramas_con_detecciones = len(
    df_fotogramas_con_detecciones
)

fotogramas_sin_detecciones = (
    total_fotogramas - fotogramas_con_detecciones
)

# 7. Construir el contexto general.
contexto_general = (
    f"El video {video_nombre} tiene una duración aproximada "
    f"de {duracion:.2f} segundos. "
    f"Se analizaron {total_fotogramas} fotogramas. "
    f"De ellos, {fotogramas_con_detecciones} presentaron "
    f"predicciones de YOLOv8n y {fotogramas_sin_detecciones} "
    f"no presentaron predicciones. "
)

if conteo_clases:
    contexto_general += (
        "Las predicciones registradas fueron: "
        + ", ".join(
            f"{clase}: {cantidad}"
            for clase, cantidad in conteo_clases.items()
            if cantidad > 0
        ) + ". "
    )

if elementos_recurrentes:
    contexto_general += (
        "En las descripciones de BLIP-2 se identificaron "
        "como elementos recurrentes: "
        + ", ".join(elementos_recurrentes) + ". "
    )

for nombre, elementos in elementos_temporales.items():
    if elementos:
        contexto_general += (
            f"En el tramo {nombre} del video se mencionan "
            + ", ".join(elementos) + ". "
        )

if coincidencias:
    contexto_general += (
        "Se encontraron coincidencias textuales específicas "
        "entre las clases detectadas por YOLOv8n y las "
        "descripciones de BLIP-2 en los mismos fotogramas. "
    )
else:
    contexto_general += (
        "No se encontraron coincidencias textuales específicas "
        "entre las clases detectadas por YOLOv8n y las "
        "descripciones de BLIP-2 en los mismos fotogramas. "
    )

if menciones_generales_armas:
    contexto_general += (
        "En algunos de esos fotogramas BLIP-2 también realizó "
        "menciones generales a armas, sin especificar "
        "necesariamente su tipo. "
    )

contexto_general += (
    "Las predicciones de YOLOv8n no constituyen una confirmación "
    "definitiva de la presencia de armas. La ausencia de "
    "predicciones tampoco demuestra que no existan armas "
    "en el video."
)

# 8. Mostrar resultados.
print("CONTEXTO GENERAL DEL VIDEO\n")
print(contexto_general)

df_contexto_general = pd.DataFrame([{
    "Video": video_nombre,
    "Duracion_segundos": duracion,
    "Fotogramas_analizados": total_fotogramas,
    "Fotogramas_con_detecciones": fotogramas_con_detecciones,
    "Fotogramas_sin_detecciones": fotogramas_sin_detecciones,
    "Predicciones_YOLO": dict(conteo_clases),
    "Elementos_recurrentes": elementos_recurrentes,
    "Coincidencias_YOLO_BLIP2": coincidencias,
    "Menciones_generales_armas": menciones_generales_armas,
    "Contexto_general": contexto_general
}])

display(df_contexto_general)

CONTEXTO GENERAL DEL VIDEO

El video video_sin_arma.webm tiene una duración aproximada de 110.98 segundos. Se analizaron 17 fotogramas. De ellos, 8 presentaron predicciones de YOLOv8n y 9 no presentaron predicciones. Las predicciones registradas fueron: pistol: 9. En las descripciones de BLIP-2 se identificaron como elementos recurrentes: tienda, gente, comprando, vista, compras, cámara, vigilancia, gran, grupo, arriba. En el tramo inicial del video se mencionan tienda, gente, vista, gran, compras. En el tramo intermedio del video se mencionan tienda, gente, comprando, vista, compras. En el tramo final del video se mencionan gente, tienda, cámara, vigilancia, comprando. No se encontraron coincidencias textuales específicas entre las clases detectadas por YOLOv8n y las descripciones de BLIP-2 en los mismos fotogramas. Las predicciones de YOLOv8n no constituyen una confirmación definitiva de la presencia de armas. La ausencia de predicciones tampoco demuestra que no existan armas en el v

,Video,Duracion_segundos,Fotogramas_analizados,Fotogramas_con_detecciones,Fotogramas_sin_detecciones,Predicciones_YOLO,Elementos_recurrentes,Coincidencias_YOLO_BLIP2,Menciones_generales_armas,Contexto_general
0,video_sin_arma.webm,110.97533,17,8,9,"{'knife': 0, 'pistol': 9}","[tienda, gente, comprando, vista, compras, cám...",[],[],El video video_sin_arma.webm tiene una duració...


In [ ]:
# 18. Exportación de resultados a Excel.
ruta_resultados = "/content/resultados_analisis_video.xlsx"
with pd.ExcelWriter(ruta_resultados, engine="openpyxl") as writer:
    df_integrado.to_excel(writer, sheet_name="Resultados_integrados", index=False)
    df_fotogramas_con_detecciones.to_excel(writer, sheet_name="Detecciones_YOLO", index=False)
    df_fotogramas.to_excel(writer, sheet_name="Fotogramas_seleccionados", index=False)
    df_contexto_general.to_excel(writer, sheet_name="Contexto_general", index=False)
    df_detecciones.to_excel(writer, sheet_name="YOLO_muestreo_video", index=False)

print("Excel guardado en:", ruta_resultados)

Excel guardado en: /content/resultados_analisis_video.xlsx


In [ ]:
# 19. Descarga del archivo Excel.
files.download(ruta_resultados)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 20. Descarga opcional de las imágenes anotadas.
# Si no hubo detecciones, no se genera un ZIP vacío.
if os.listdir(output_boxes):
    archivo_zip = shutil.make_archive(
        "/content/fotogramas_con_detecciones", "zip", output_boxes
    )
    files.download(archivo_zip)
else:
    print("No hay imágenes anotadas para descargar.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>